In [ ]:
#!/usr/bin/env python3
import os
import itertools as it
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# Directories
FILE_PATH = "/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/BDR_AD_control_merged_ml.raw"

PHENO_COL = "PHENOTYPE_x"           
PC_COLS   = [f"PC{i}" for i in range(1, 8)]  

SNP_COLS = None

OUT_DIR = "/scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/pairwise_interaction"
os.makedirs(OUT_DIR, exist_ok=True)

# Output file names
ALL_CSV        = os.path.join(OUT_DIR, "pairwise_interactions_all.csv")
SIG_BONF_CSV   = os.path.join(OUT_DIR, "pairwise_interactions_sig_bonf.csv")
SIG_FDR_BH_CSV = os.path.join(OUT_DIR, "pairwise_interactions_sig_fdr_bh.csv")

UNCORR_ALPHA = 0.05
UNCORR_DIR   = os.path.join(OUT_DIR, "uncorrected_sig")
os.makedirs(UNCORR_DIR, exist_ok=True)
UNCORR_SIG_CSV = os.path.join(UNCORR_DIR, "pairwise_interactions_sig_raw_p.csv")

# GLM settings
MAXITER   = 200
ROBUST_SE = True

# Correction thresholds
ALPHA_FWER = 0.05   # Bonferroni
ALPHA_FDR  = 0.05   # Benjamini–Hochberg


def load_and_prepare(path: str):
    """
    Load PLINK-like .raw / merged file, normalize phenotype to 0/1,
    auto-detect SNP columns if needed, drop missing across required cols.
    """
    # Read whitespace-delimited file
    df = pd.read_csv(path, sep=r"\s+", engine="python")

    # Clean headers
    df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

    if PHENO_COL not in df.columns:
        raise KeyError(f"{PHENO_COL!r} not found. First few columns: {df.columns[:10].tolist()}")

    # Normalise phenotype to 0/1
    y_raw = df[PHENO_COL]
    uniq = set(pd.unique(y_raw.dropna()))
    if uniq == {1, 2} or uniq == {1.0, 2.0}:
        # Map (Control=1, Case=2) to (0,1)
        y = y_raw.replace({2: 1, 1: 0}).astype(int)
    elif uniq.issubset({0, 1}) or uniq.issubset({0.0, 1.0}):
        y = y_raw.astype(int)
    else:
        raise ValueError(f"{PHENO_COL} must be 0/1 or {{1,2}}; got {sorted(list(uniq))!r}")
    df[PHENO_COL] = y

    # PCs present
    pcs_present = [pc for pc in PC_COLS if pc in df.columns]

    # Detect SNP columns if not provided
    snp_cols = SNP_COLS
    if snp_cols is None:
        exclude = set([PHENO_COL] + pcs_present + ["FID", "IID", "PAT", "MAT", "SEX",
                                                   "PRS", "KEY", "PHENOTYPE_y"])
        candidate_cols = [c for c in df.columns if c not in exclude]
        snp_cols = []
        for c in candidate_cols:
            if pd.api.types.is_numeric_dtype(df[c]):
                vals = df[c].dropna().unique()
                # treat dosage/genotype columns with values 0/1/2 as SNPs
                if len(vals) <= 3 and set(np.round(vals).astype(int)).issubset({0, 1, 2}):
                    snp_cols.append(c)

        if len(snp_cols) == 0:
            raise ValueError("No SNP columns detected. Set SNP_COLS explicitly.")

    # Drop rows with missing phenotype/PCs/SNPs; enforce numeric types
    needed = [PHENO_COL] + pcs_present + snp_cols
    df = df.dropna(subset=needed).copy()

    for c in snp_cols + pcs_present:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=snp_cols + pcs_present)

    return df, snp_cols, pcs_present


def fit_logit_interaction(y, s1, s2, pcs_dict):
    """
    Logistic model: y ~ s1 + s2 + s1*s2 + PCs
    Returns beta/SE/pval for interaction term, its OR, and convergence info.
    """
    X = pd.DataFrame({
        "SNP1": s1,
        "SNP2": s2,
        "INT": s1 * s2,
    })
    for pc_name, pc_vals in pcs_dict.items():
        X[pc_name] = pc_vals
    X = sm.add_constant(X, has_constant="add")

    try:
        model = sm.GLM(y, X, family=sm.families.Binomial())
        if ROBUST_SE:
            res = model.fit(maxiter=MAXITER, cov_type="HC3")
        else:
            res = model.fit(maxiter=MAXITER)

        beta = res.params["INT"]
        se   = res.bse["INT"]
        pval = res.pvalues["INT"]
        OR   = float(np.exp(beta))
        return beta, se, pval, OR, True, None
    except Exception as e:
        return np.nan, np.nan, np.nan, np.nan, False, str(e)


def main():
    df, snp_cols, pcs_present = load_and_prepare(FILE_PATH)
    y = df[PHENO_COL].astype(int).values
    pcs = {pc: df[pc].values for pc in pcs_present}

    pairs = list(it.combinations(snp_cols, 2))
    n_pairs_total = len(pairs)
    print(f"Fitting {n_pairs_total} pairwise interaction models across {len(snp_cols)} SNPs "
          f"with {len(pcs_present)} PCs...")

    rows = []
    for k, (a, b) in enumerate(pairs, start=1):
        beta, se, pval, OR, converged, err = fit_logit_interaction(
            y=y, s1=df[a].values, s2=df[b].values, pcs_dict=pcs
        )
        rows.append({
            "SNP_A": a,
            "SNP_B": b,
            "beta_INT": beta,
            "se_INT": se,
            "OR_INT": OR,
            "pval_INT": pval,
            "converged": converged,
            "error": err,
            "N": len(y)
        })
        if k % 200 == 0 or k == n_pairs_total:
            print(f"  ... {k}/{n_pairs_total} pairs processed")

    res = pd.DataFrame(rows)

    # Multiple-testing corrections
    # Bonferroni (FWER)
    p_bonf = np.minimum(pvals * n_valid, 1.0)

    # Benjamini–Hochberg (FDR)
    rej_bh, q_bh, _, _ = multipletests(pvals, alpha=ALPHA_FDR, method="fdr_bh")

    # Attach corrected columns
    res["pval_bonf"] = np.nan
    res["qval_bh"]   = np.nan
    res["rej_bh"]    = False
    res.loc[valid_mask, "pval_bonf"] = p_bonf
    res.loc[valid_mask, "qval_bh"]   = q_bh
    res.loc[valid_mask, "rej_bh"]    = rej_bh

    # Save: full table sorted by raw p
    res_all = res.sort_values(["pval_INT"], na_position="last")
    res_all.to_csv(ALL_CSV, index=False)
    print(f"[WRITE] All results -> {ALL_CSV}  (n={len(res_all)})")

    # Save uncorrected significant interactions (raw p ≤ UNCORR_ALPHA)
    res_uncorr = (
        res.loc[res["pval_INT"].notna() & (res["pval_INT"] <= UNCORR_ALPHA)]
           .sort_values(["pval_INT"])
    )
    res_uncorr.to_csv(UNCORR_SIG_CSV, index=False)
    print(f"[WRITE] Uncorrected significant (p <= {UNCORR_ALPHA}) -> {UNCORR_SIG_CSV}  (n={len(res_uncorr)})")

    # Save Bonferroni significant
    res_bonf = res.loc[res["pval_bonf"] <= ALPHA_FWER].sort_values(["pval_bonf", "pval_INT"])
    res_bonf.to_csv(SIG_BONF_CSV, index=False)
    print(f"[WRITE] Bonferroni significant (alpha={ALPHA_FWER}) -> {SIG_BONF_CSV}  (n={len(res_bonf)})")

    # Save BH-FDR significant
    res_bh = res.loc[res["rej_bh"]].sort_values(["qval_bh", "pval_INT"])
    res_bh.to_csv(SIG_FDR_BH_CSV, index=False)
    print(f"[WRITE] BH-FDR significant (alpha={ALPHA_FDR}) -> {SIG_FDR_BH_CSV}  (n={len(res_bh)})")


main()


Fitting 2485 pairwise interaction models across 71 SNPs with 7 PCs...


/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/

  ... 200/2485 pairs processed


/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/

  ... 400/2485 pairs processed


...

/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/

  ... 2400/2485 pairs processed


/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/

  ... 2485/2485 pairs processed
[WRITE] All results -> /scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/pairwise_interaction/pairwise_interactions_all.csv  (n=2485)
[WRITE] Uncorrected significant (p <= 0.05) -> /scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/pairwise_interaction/uncorrected_sig/pairwise_interactions_sig_raw_p.csv  (n=20)
[WRITE] Bonferroni significant (alpha=0.05) -> /scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/pairwise_interaction/pairwise_interactions_sig_bonf.csv  (n=0)
[WRITE] BH-FDR significant (alpha=0.05) -> /scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/pairwise_interaction/pairwise_interactions_sig_fdr_bh.csv  (n=0)


/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:576: RuntimeWarning: invalid value encountered in multiply
  tmp = score_factor * tmp
/scratch/c.c2029098/ml_envs/ml_env/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:257: RuntimeWarning: divide by zero encountered in divide
  return (2 * p - 1) / v ** 2
/scratch/c.c2029098/ml_envs/ml_env/lib/